[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somanshusingla/llm-from-scratch/blob/main/notebooks/02_ch3_attention_mechanisms.ipynb)

# Chapter 3 — Coding Attention Mechanisms

**Build a Large Language Model (From Scratch)** · notebook 2 of 7

Attention is the single idea that makes transformers work. This notebook builds
it up in four stages, each one a small step from the last:

1. **Simplified self-attention** — no learnable weights, just to get the intuition.
2. **Self-attention with trainable weights** — the real scaled dot-product attention.
3. **Causal (masked) attention** — stop a token from "looking at the future".
4. **Multi-head attention** — run several attentions in parallel (what GPT uses).

The final `MultiHeadAttention` class you write here is dropped, unchanged, into
the GPT model in Chapter 4.

> **Mental model:** attention lets each token look at every other token and pull
> in the information it needs, weighted by relevance. "The animal didn't cross
> the street because *it* was tired" — attention is how "it" figures out it means
> "the animal".

*Pure tensor math — runs instantly on CPU.*

In [ ]:
import torch
import torch.nn as nn
print("torch:", torch.__version__)

torch: 2.11.0+cu128


## 3.1 The problem with modeling long sequences

Before attention, people used RNNs, which read text one token at a time and
squeeze everything into a single hidden state. For long sequences that state
becomes a bottleneck — early information gets lost. Attention removes the
bottleneck: every token gets **direct access** to every other token.

## 3.2 Capturing data dependencies with attention

We'll represent a 6-token sentence — *"Your journey starts with one step"* — as
6 vectors of dimension 3. (In a real model these come from the embedding pipeline
you built in Chapter 2; here we hand-pick small numbers so you can see the math.)

In [ ]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89],  # Your     (x^1)
   [0.55, 0.87, 0.66],  # journey  (x^2)
   [0.57, 0.85, 0.64],  # starts   (x^3)
   [0.22, 0.58, 0.33],  # with     (x^4)
   [0.77, 0.25, 0.10],  # one      (x^5)
   [0.05, 0.80, 0.55]]  # step     (x^6)
)
print("inputs shape:", inputs.shape)   # (6 tokens, 3 dims)

inputs shape: torch.Size([6, 3])


## 3.3 Attending to different parts of the input with self-attention

**Goal:** compute a *context vector* for each token — a weighted sum of all
tokens, where the weights say "how much should this token pay attention to that
one?"

### Step A: attention scores
The simplest relevance measure between two vectors is their **dot product**.
Let's compute how much token 2 ("journey") relates to every token.

In [ ]:
query = inputs[1]  # the 2nd token, "journey"

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)

print("Raw attention scores for token 2:", attn_scores_2)

Raw attention scores for token 2: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### Step B: normalize to weights (softmax)
`softmax` turns the raw scores into positive weights that sum to 1 — a
probability distribution over "where to look".

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


### Step C: the context vector
Multiply each input by its weight and add them up. The result is a new vector for
token 2 that blends in information from the whole sentence.

In [ ]:
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print("Context vector for token 2:", context_vec_2)

Context vector for token 2: tensor([0.4419, 0.6515, 0.5683])


### All tokens at once (with matrix multiplication)
The loops above are just to build intuition. In practice we do all tokens
simultaneously with one matrix multiply — this is why GPUs love transformers.

In [ ]:
attn_scores = inputs @ inputs.T          # (6,6): every token vs every token
attn_weights = torch.softmax(attn_scores, dim=-1)
all_context_vecs = attn_weights @ inputs # (6,3): a context vector per token

print("attn_weights shape:", attn_weights.shape)
print("context vectors shape:", all_context_vecs.shape)
print("row 2 matches the loop version:", torch.allclose(all_context_vecs[1], context_vec_2))

attn_weights shape: torch.Size([6, 6])
context vectors shape: torch.Size([6, 3])
row 2 matches the loop version: True


## 3.4 Implementing self-attention with trainable weights

Real attention isn't parameter-free. Each input is projected into three roles via
learnable weight matrices:

- **Query (Q)** — "what am I looking for?"
- **Key (K)** — "what do I offer?"
- **Value (V)** — "what information do I carry?"

Attention score = Q · K. We divide by `sqrt(head_dim)` (**scaled** dot-product)
to keep gradients stable, then softmax, then weight the Values.

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        attn_scores = queries @ keys.T                       # (T, T)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)        # scaled + softmax
        context_vec = attn_weights @ values                  # (T, d_out)
        return context_vec


torch.manual_seed(789)
sa = SelfAttention_v2(d_in=3, d_out=2)
print(sa(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## 3.5 Hiding future words with causal attention

GPT predicts the *next* token, so when processing position `t` it must **not**
peek at positions after `t` — otherwise it would cheat. We enforce this with a
mask: set the scores for future positions to `-inf` before softmax, so their
weight becomes 0.

We also add **dropout** on the attention weights (a regularizer that randomly
zeros some weights during training), and support **batches** of sequences.

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # a fixed upper-triangular mask (1s above the diagonal = "the future")
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)          # (b, T, T)
        attn_scores.masked_fill_(                             # block the future
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec


# Simulate a batch of 2 identical sequences.
batch = torch.stack((inputs, inputs), dim=0)
print("batch shape:", batch.shape)   # (2, 6, 3)

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in=3, d_out=2, context_length=context_length, dropout=0.0)
context_vecs = ca(batch)
print("context_vecs shape:", context_vecs.shape)  # (2, 6, 2)

batch shape: torch.Size([2, 6, 3])
context_vecs shape: torch.Size([2, 6, 2])


## 3.6 Extending single-head attention to multi-head attention

One attention "head" learns one kind of relationship. Transformers run several
heads in parallel so the model can attend to different things at once (syntax,
coreference, topic, ...), then concatenate the results.

The efficient implementation below does all heads in a single set of matrix
multiplies by reshaping the projections into `(batch, heads, tokens, head_dim)`.
**This exact class is the one used in the GPT model in Chapter 4.**

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads          # split dim across heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)     # combines the heads
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)

        # (b, T, d_out) -> (b, T, num_heads, head_dim)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # -> (b, num_heads, T, head_dim)
        keys    = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values  = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)          # per-head scores
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # (b, num_heads, T, head_dim) -> (b, T, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)
        # concat heads back into d_out
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec


torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, dropout=0.0, num_heads=2)
context_vecs = mha(batch)
print("output shape:", context_vecs.shape)   # (2, 6, 2)
print(context_vecs)

output shape: torch.Size([2, 6, 2])
tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)


## Summary

You built attention from the ground up:

- **Scores** via dot products → **weights** via softmax → **context vectors** via
  weighted sums of Values.
- **Q/K/V** projections make it learnable and **scaling** keeps it stable.
- A **causal mask** prevents peeking at the future (required for text generation).
- **Multiple heads** let the model attend to several patterns at once.

The `MultiHeadAttention` class you just wrote is a real, GPT-grade component.

### Try it yourself
- Set `num_heads=1` and compare with the `CausalAttention` output shape.
- Print `attn_weights` inside `forward` and confirm every row's upper triangle is 0.

**Next:** `03_ch4_gpt_model.ipynb` — assemble attention + feed-forward + layer
norm into a full GPT-2 model.